# Collision-Aware Social-STGCNN — Google Colab Training

Runs our collision-aware extension of Social-STGCNN on Colab GPUs.

**Before running:** Runtime → Change runtime type → GPU (T4 / L4 / A100).

Sections:
1. Environment setup (install deps, check GPU)
2. Get the code (Google Drive mount OR GitHub clone OR file upload)
3. Evaluate pretrained baselines (Finding 1)
4. Train a single collision-aware model
5. Train Phase 1 (all 10 experiments)
6. Evaluate Phase 1 checkpoints
7. Copy checkpoints back to Google Drive

## 1. Environment setup

In [ ]:
# Check GPU — should print NVIDIA info.
!nvidia-smi

In [ ]:
# PyTorch comes preinstalled on Colab with CUDA. Verify:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Install/upgrade auxiliary deps. networkx>=3 is required (from_numpy_array).
!pip install -q 'networkx>=3.0' 'tqdm>=4.60'

## 2. Get the code

Pick **ONE** of the three options below. The other two cells can be ignored.

### Option A — Mount Google Drive (recommended)

Put the repo at `MyDrive/DLproject-Social-STGCNN/` in your Drive. Checkpoints will persist across Colab sessions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/DLproject-Social-STGCNN'
assert os.path.isdir(PROJECT_DIR), f'Project not found at {PROJECT_DIR}. Upload the folder first.'
os.chdir(PROJECT_DIR)
print('CWD:', os.getcwd())
!ls

### Option B — Clone from GitHub

Replace the URL with your group's private repo (you'll need a PAT for private repos).

In [ ]:
# Example — edit URL for your repo.
# !git clone https://github.com/YOUR_USER/YOUR_REPO.git Social-STGCNN
# %cd Social-STGCNN

### Option C — Upload a ZIP of the project

Zip the local project folder, then run this cell and pick the zip from your machine.

In [ ]:
# from google.colab import files
# uploaded = files.upload()
# import zipfile, os
# for name in uploaded:
#     with zipfile.ZipFile(name) as z:
#         z.extractall('/content/')
# os.chdir('/content/DLproject-Social-STGCNN')  # adjust to actual extracted folder
# !ls

## 3. Evaluate pretrained baselines

Produces the motivating finding (collision rates on pretrained models).

In [ ]:
!python test.py

## 4. Train ONE collision-aware model

Use this for quick experimentation / debugging. Edit the values below.

In [ ]:
SCENE        = 'eth'          # eth | hotel | univ | zara1 | zara2
LOSS         = 'ecp'          # hinge | exp | inv | gauss | ecp | none
LAMBDA       = 1.0
D_MIN        = 0.4
EPOCHS       = 250
TAG          = f'{SCENE}-{LOSS}-lam{LAMBDA}-dmin{D_MIN}'

!python train.py \
    --dataset {SCENE} \
    --tag {TAG} \
    --collision_loss {LOSS} \
    --lambda_col {LAMBDA} \
    --d_min {D_MIN} \
    --lr 0.01 --n_stgcnn 1 --n_txpcnn 5 \
    --num_epochs {EPOCHS} --use_lrschd

## 5. Phase 1 — run all 10 experiments

5 losses x 2 scenes (ETH sparse + UNIV dense) = 10 runs. Estimated ~8 hours on T4.

Colab Pro sessions can last ~12 hours. If your session is shorter, use `--losses` / `--scenes` to split this into multiple sessions — checkpoints in Drive persist.

In [ ]:
# Full Phase 1 (will take ~8h).
!python run_phase1.py

In [ ]:
# OR run a subset (e.g. if you have limited time in one session):
# !python run_phase1.py --scenes eth --losses hinge ecp
# !python run_phase1.py --epochs 50  # quick smoke test

## 6. Evaluate Phase 1

Produces `phase1_results.txt` with ADE / FDE / ColRate metrics for every checkpoint.

In [ ]:
!python run_phase1_eval.py

## 7. Persist checkpoints

If you used Option A (Drive), checkpoints are already persisted. If you used Options B/C, copy them back to Drive now.

In [ ]:
# Only needed if repo lives in /content/ (Options B/C)
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r ./checkpoint /content/drive/MyDrive/DLproject-Social-STGCNN-checkpoints/
# !cp phase1_results.txt /content/drive/MyDrive/DLproject-Social-STGCNN-checkpoints/